In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

R = Path("results/latest")  
solves = pd.read_csv(R / "solves.csv")
pw = pd.read_csv(R / "pairwise.csv")
t = pd.read_csv(R / "timings.csv").merge(
    solves[["solve_id","solver_name","error_type","generation_method","feasible"]], on="solve_id")

COLORS = {"SNOPT": "green", "Ipopt": "purple", "NLopt": "orange"}

In [ ]:
# feas rate
finite = solves[solves.error_type.isna()] # no errors + no nan

fig, ax = plt.subplots(figsize=(7, 4.5))
for solver in ("SNOPT", "Ipopt", "NLopt"):
    a = solves[solves.solver_name == solver]
    f = finite[finite.solver_name == solver]
    if a.empty or f.empty:
        continue
    ax.scatter(f.time_median.median(), 
               a.feasible.mean(),               # proportion feasible
               s=160, 
               color=COLORS[solver], 
               edgecolor="black", 
               linewidth=0.7)
    ax.annotate(solver, 
                (f.time_median.median(), 
                 a.feasible.mean()),
                textcoords="offset points", 
                xytext=(8, 4),
                fontsize=9, 
                color=COLORS[solver])
ax.set_xlabel("median successful solve time (ms)")
ax.set_ylabel("feasibility rate")
ax.set_ylim(0, 1)
ax.set_xlim(0, 0.06)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
d = pw[pw.both_feasible]

tols = sorted(d.orientation_tol.unique())
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
d = pw[pw.both_feasible]

for ax, col, title in [(ax1, "common_residual_inf_diff", "optimality difference"),
                       (ax2, "cost_diff_abs", "cost difference")]:
    groups = [d.loc[d.orientation_tol == t, col].abs().dropna() for t in tols]
    groups = [g[g > 0] for g in groups]         
    ax.boxplot(groups)
    ax.set_xticklabels([str(t) for t in tols])
    ax.set_yscale("log")
    ax.set_xlabel("orientation_tol"); ax.set_ylabel(col); ax.set_title(title)

fig.tight_layout(); plt.show()